# TaaS Studio — Hypergraph Explorer Browser E2E (existing Colab Studio)

**Prerequisite:** run the branch-pinned `taas_core_studio_colab.ipynb` from this same PR branch, top-to-bottom, including Launch Studio and Open TaaS Studio. This add-on reuses its exact process, token and proxy. It fetches only the browser test module into the existing checkout; it never checks out another revision or starts another server.

The older `main`-branch core notebook pins a pre-Explorer commit and its checkout contains the stale test source that caused the SyntaxError. Do not use that older runtime for this E2E validation.


In [ ]:
# Reuse the running Studio launched by taas_core_studio_colab.ipynb.
# Do not start another API process or create/replace its token.
import os, pathlib, subprocess
from IPython.display import display, Image

WORKDIR = pathlib.Path("/content/taas-provenance-discovery")
assert WORKDIR.is_dir(), "Run taas_core_studio_colab.ipynb first in this same Colab runtime."
assert (WORKDIR / "src" / "taas" / "studio" / "api.py").is_file()
assert "studio_process" in globals(), "The TaaS Core Studio launch cell must run first."
assert studio_process.poll() is None, "Studio is not running. Re-run its Launch Studio cell."
assert "api_token" in globals() and api_token, "The Core Studio notebook token is unavailable."
assert "proxy_url" in globals() and proxy_url, "Run the Open TaaS Studio cell first."

TAAS_STUDIO_URL = proxy_url
os.environ["TAAS_STUDIO_URL"] = TAAS_STUDIO_URL
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token

os.environ["TAAS_E2E_ARTIFACT_DIR"] = "/content/taas-e2e-artifacts"
# Synchronize only the browser test module from this PR branch. The live Studio
# process, API token, checkout revision and Colab proxy are not replaced.
branch = "feature/hypergraph-explorer-browser-e2e"
subprocess.run(["git", "fetch", "origin", branch], cwd=WORKDIR, check=True)
test_source = subprocess.check_output(["git", "show", f"origin/{branch}:tests/test_studio_browser_e2e.py"], cwd=WORKDIR, text=True)
(WORKDIR / "tests" / "test_studio_browser_e2e.py").write_text(test_source)
print("Browser E2E test module synchronized; existing Studio process/token/proxy unchanged.")
print("Reusing Studio process PID:", studio_process.pid)
print("Existing proxy URL:", TAAS_STUDIO_URL)
print("Existing API token: present (value hidden)")
%cd /content/taas-provenance-discovery



In [ ]:
!pip -q install playwright
!playwright install --with-deps chromium
!python -m compileall -q src/taas tests/test_studio_browser_e2e.py


In [ ]:
!pytest -q tests/test_retrieval_degenerate.py tests/test_studio_hypergraph_explorer.py tests/test_studio_navigation_shell.py tests/test_studio_professional_ux.py tests/test_hypergraph_projection.py tests/test_engine_certificate_gate.py tests/test_studio_browser_e2e.py -s


In [ ]:
from pathlib import Path
from IPython.display import display, Image

artifact_dir = Path(os.environ.get('TAAS_E2E_ARTIFACT_DIR', '/content/taas-e2e-artifacts'))
artifacts = sorted(artifact_dir.glob('hypergraph-explorer-*.png')) if artifact_dir.exists() else []
if not artifacts:
    print('No browser screenshots were exported.')
    print('Artifact directory:', artifact_dir)
    print('Inspect the pytest cell first: screenshots are produced only after each viewport test reaches its screenshot step.')
    raise RuntimeError('Browser screenshot artifacts are unavailable; resolve the pytest failure/skip before displaying images.')
for artifact in artifacts:
    assert artifact.is_file() and artifact.stat().st_size > 0, f'Invalid screenshot artifact: {artifact}'
    print(f'{artifact.name} — {artifact.stat().st_size:,} bytes')
    display(Image(filename=str(artifact)))


## Runtime reuse and validation boundary

- Reuses server process, token and proxy URL created by `taas_core_studio_colab.ipynb`.
- Does not reinstall TaaS, create a token, launch Uvicorn or call `proxyPort`.
- Chromium visits the existing Studio proxy URL and authenticates with the existing notebook token.
- Discovery must return `taas.hypergraph-projection/v1` and `execution_performed=false`.
- Exercises search, SVG keyboard selection, accessible entity list and inspector.
- Runs desktop and mobile viewport checks and produces screenshot artifacts.
- Asserts no `/api/v1/mcp/*` browser requests.

A pass validates this browser journey against the current Colab process only. It does not prove production identity, live MCP execution, authorization, cross-browser compatibility, screen-reader certification or deployment resilience. Keep Colab output distinct from GitHub Actions status.
